# 🎙️ OmniASR-CTC-1B — Tunisian Arabic ASR Benchmark

**Model:** facebook/omniASR-CTC-1B  
**Config key:** omniasr_ctc_1b

> OmniASR uses the omnilingual-asr package (fairseq2-based), NOT HF Transformers.  
> Inference goes through ASRInferencePipeline with Arabic language tag ara_Arab.  
> Audio is passed as [{'waveform': array, 'sample_rate': sr}] dicts.  
> Constraint: segments must be ≤ 40 s; longer audio must be pre-chunked.

## 1 · Install & Imports

In [ ]:
# SETUP INSTRUCTIONS FOR LOCAL SERVER
# =====================================
# omnilingual-asr has broken dependencies on PyPI (fairseq2n missing).
# Install manually before running this notebook:
#
# conda create -n omni python=3.10 -y
# conda activate omni
# pip install --no-deps omnilingual-asr
# pip install datasets evaluate jiwer pyyaml torch numpy
# 
# Then select kernel: "Python 3.10 (omni)" and run this notebook.
# ===================================================================

# If packages are already installed, comment out this cell and proceed.
# !pip install -q omnilingual-asr datasets evaluate jiwer pyyaml
# !apt-get install -y -q libsndfile1  # For audio processing on Linux

ERROR: Cannot install fairseq2 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts

[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# Local path to benchmarking package
import sys
from pathlib import Path

BENCHMARK_ROOT = Path('/home/ala/dataset')
assert BENCHMARK_ROOT.exists(), f'Missing path: {BENCHMARK_ROOT}'

if str(BENCHMARK_ROOT) not in sys.path:
    sys.path.insert(0, str(BENCHMARK_ROOT))

print('Using benchmark root:', BENCHMARK_ROOT)

Using benchmark root: /home/ala/dataset


In [4]:
from benchmark_utils import (
    load_config, get_device, print_gpu_info, setup_output_dir,
    load_benchmark, split_benchmark,
    compute_metrics, per_sample_wer,
    run_pipeline_inference, build_results_df,
    run_labelled_splits, run_unlabelled_splits,
    display_preview, display_worst, display_bulk_predictions,
    audio_inspector, display_summary, plot_wer_cer,
)
import numpy as np
import torch
import time
from pathlib import Path
from tqdm.auto import tqdm
from datasets import Audio as HFAudio

cfg = load_config(str(BENCHMARK_ROOT / 'config.yaml'))

# Override Colab-only paths for local execution
cfg['paths']['dataset'] = str(BENCHMARK_ROOT)
cfg['paths']['output_root'] = str(Path('/home/ala/TunisianDialogSystem/outputs/asr_benchmark_results'))

TARGET_SR    = cfg['evaluation']['target_sr']
TOP_N_WORST  = cfg['evaluation']['top_n_worst']
PREVIEW_ROWS = cfg['evaluation']['preview_rows']
RESUME = False

print('Dataset path:', cfg['paths']['dataset'])
print('Output root :', cfg['paths']['output_root'])

Dataset path: /home/ala/dataset
Output root : /home/ala/TunisianDialogSystem/outputs/asr_benchmark_results


## 2 · GPU Check

In [5]:
device = get_device()
print_gpu_info()

Device : cuda
GPU    : NVIDIA GB10
VRAM   : 128.5 GB total  |  128.5 GB free


## 3 · Load Model & Pipeline

### 3.1 Load `ASRInferencePipeline` (`omniASR_CTC_1B`)
`ASRInferencePipeline` downloads and caches the model weights on first call.
Pass `model_card` exactly as listed in the omnilingual-asr registry.

| Parameter | Value | Note |
|-----------|-------|--------|
| `model_card` | `omniASR_CTC_1B` | Registry name |  
| `language` | `ara_Arab` | BCP-47 + script |  
| `batch_size` | from config | Tune to GPU VRAM |



In [6]:
!pip install torchaudio retrying

In [7]:
import sys
sys.path.insert(0, '/home/ala/fairseq2/src')
sys.path.insert(0, '/home/ala/fairseq2/native/python/src')

from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

In [8]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

mcfg        = cfg['models']['omniasr_ctc_1b']
OUTPUT_DIR  = setup_output_dir(cfg, 'omniasr_ctc_1b')
FILE_SUFFIX = 'omniasr_ctc_1b_results'
BATCH_SIZE  = mcfg['batch_size']
LANG        = mcfg['language']

print("Loading omniASR_CTC_1B ...")
omni_pipe = ASRInferencePipeline(model_card=mcfg['model_card'])
print("✓ omniASR_CTC_1B loaded")

Loading omniASR_CTC_1B ...


Output()

✓ omniASR_CTC_1B loaded


## 4 · Mount Drive & Load Benchmark

In [9]:
from datasets import DatasetDict, load_from_disk

# Build a DatasetDict from available on-disk split folders only
split_dirs = sorted([p for p in BENCHMARK_ROOT.iterdir() if p.is_dir()])
benchmark = DatasetDict()

for p in split_dirs:
    try:
        benchmark[p.name] = load_from_disk(str(p))
    except Exception:
        pass

print(f"Loaded splits: {list(benchmark.keys())}")
LABELLED_SPLITS, UNLABELLED_SPLITS = split_benchmark(benchmark)

Loaded splits: ['labeled_algerian', 'labeled_linagora_cs_arabize', 'labeled_linagora_raw', 'unlabeled_youtube']
Labelled splits   : ['labeled_algerian', 'labeled_linagora_cs_arabize', 'labeled_linagora_raw']
Unlabelled splits : ['unlabeled_youtube']


## 5 · Inference

In [22]:
!pip install torchcodec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 10.6 MB/s  0:00:00m0:00:01


In [10]:
import sys
print(sys.executable)
import torchcodec
print('torchcodec', torchcodec.__version__)
import torch
print('torch', torch.__version__, 'cuda_available=', torch.cuda.is_available())

/home/ala/miniconda3/envs/omni-py310/bin/python
torchcodec 0.11.1+cu130
torch 2.11.0+cu130 cuda_available= True


In [12]:
RESUME = False

# OmniASR hard limit: <= 40 seconds per segment
MAX_ASR_SEC = 40.0
CHUNK_SEC = 39.5  # safety margin
CHUNK_SAMPLES = int(TARGET_SR * CHUNK_SEC)

def _split_audio_for_omniasr(waveform: np.ndarray, sample_rate: int = TARGET_SR):
    """Return a list of ASR input dicts, each <= 40s."""
    if waveform is None or len(waveform) == 0:
        return [{'waveform': np.zeros(1, dtype=np.float32), 'sample_rate': sample_rate}]

    max_allowed = int(MAX_ASR_SEC * sample_rate)
    if len(waveform) <= max_allowed:
        return [{'waveform': waveform.astype(np.float32), 'sample_rate': sample_rate}]

    chunks = []
    for start in range(0, len(waveform), CHUNK_SAMPLES):
        seg = waveform[start:start + CHUNK_SAMPLES]
        if len(seg) == 0:
            continue
        chunks.append({'waveform': seg.astype(np.float32), 'sample_rate': sample_rate})
    return chunks


def infer_fn(ds):
    """
    Wraps ASRInferencePipeline.transcribe().
    Input format: [{'waveform': np.ndarray, 'sample_rate': int}]
    Language tag: 'ara_Arab'.
    Handles OmniASR 40s limit by auto-chunking long audio.
    """
    ds = ds.cast_column('audio', HFAudio(sampling_rate=TARGET_SR))
    predictions, latencies = [], []
    start = time.time()

    for i in tqdm(range(0, len(ds), BATCH_SIZE), desc='Inferring', unit='batch'):
        batch = ds.select(range(i, min(i + BATCH_SIZE, len(ds))))

        # Build per-sample chunk lists
        chunked_per_sample = []
        for x in batch:
            wav = np.array(x['audio']['array'], dtype=np.float32)
            chunked_per_sample.append(_split_audio_for_omniasr(wav, TARGET_SR))

        # Flatten for one transcribe call
        flat_audio_data = [chunk for sample_chunks in chunked_per_sample for chunk in sample_chunks]

        t0 = time.time()
        flat_texts = omni_pipe.transcribe(
            flat_audio_data,
            lang=[LANG] * len(flat_audio_data),
            batch_size=BATCH_SIZE,
        )
        elapsed = time.time() - t0

        # Recompose chunk texts back to one text per original sample
        pos = 0
        for sample_chunks in chunked_per_sample:
            n = len(sample_chunks)
            sample_text = ' '.join(t.strip() for t in flat_texts[pos:pos + n] if isinstance(t, str)).strip()
            predictions.append(sample_text)
            # Keep one latency value per original sample
            latencies.append(elapsed / max(1, len(chunked_per_sample)))
            pos += n

    return predictions, latencies, time.time() - start


all_result_dfs, summary_rows = run_labelled_splits(
    benchmark,
    LABELLED_SPLITS,
    infer_fn,
    OUTPUT_DIR,
    FILE_SUFFIX,
    PREVIEW_ROWS,
    TOP_N_WORST,
    resume=RESUME,
)


  Running: labeled_algerian  (500 samples)


Inferring:   0%|          | 0/63 [00:00<?, ?batch/s]

  WER              : 0.9823
  CER              : 0.9010
  RTF              : 0.0070
  Total audio      : 3.987 h
  Inference time   : 101.1 s
  Mean latency/smp : 0.1909 s
  Median latency   : 0.1848 s
  ✓ CSV saved → /home/ala/TunisianDialogSystem/outputs/asr_benchmark_results/omniasr_ctc_1b_results/labeled_algerian_omniasr_ctc_1b_results.csv

  Running: labeled_linagora_cs_arabize  (5482 samples)


Inferring:   0%|          | 0/686 [00:00<?, ?batch/s]

  WER              : 0.6906
  CER              : 0.3263
  RTF              : 0.0139
  Total audio      : 6.015 h
  Inference time   : 300.8 s
  Mean latency/smp : 0.0537 s
  Median latency   : 0.0531 s
  ✓ CSV saved → /home/ala/TunisianDialogSystem/outputs/asr_benchmark_results/omniasr_ctc_1b_results/labeled_linagora_cs_arabize_omniasr_ctc_1b_results.csv

  Running: labeled_linagora_raw  (5482 samples)


Inferring:   0%|          | 0/686 [00:00<?, ?batch/s]

  WER              : 0.6831
  CER              : 0.3456
  RTF              : 0.0143
  Total audio      : 6.015 h
  Inference time   : 310.4 s
  Mean latency/smp : 0.0554 s
  Median latency   : 0.0544 s
  ✓ CSV saved → /home/ala/TunisianDialogSystem/outputs/asr_benchmark_results/omniasr_ctc_1b_results/labeled_linagora_raw_omniasr_ctc_1b_results.csv

✅ All labelled splits done.


## 6 · Per-sample preview

In [13]:
display_preview(all_result_dfs, PREVIEW_ROWS)


── labeled_algerian ──


,reference,prediction,sample_wer,duration_s
0,في الدار البيضاء ومقاولات كبيرة بسبب الضبابية بسبب هاد الفراغ لي كاين ليوما على مستوى الاقتصاد السياسي واللي دخلنا في الركود التضخمي يعني التضخم دي اقتصادي هابط التراجع ديال النمو والشركات د القطاع الخصوصي الكبيرة كتكمن تقنن النفقات ديالها نقتاسم التوظيف وبالتالي يعني واحد السياسة احترازية,بيضا ومقاولت بير سبب الضبابي الفرا علىمستوى القتصاد السياس الرد تضمتضم تصاترا م ور قا الصوصي ابر تق النفقات ها قسضيف وا سيستاي,0.978,30.000000
1,في ولا ولا هذا ولا هادشي لي كنت كنقرا وليت راسي كنتوجه الذكاء المالي التطور الشخصي انا دابا انا دابا بديت في التطور الشخصي قريت عليه وبديت كنت كنقرا وبغيت ندير التطور الشخصي ماشي باش نحل مشاكل الناس اللي مجوجين ديالك اه بغيت بغيت التطور الشخصي نخدمو فالتطور,و ا ال ارت عل نر التطور الشي حل ا الن التطور الش م لتر ا,0.958,28.650000
2,تكون التجربة ما عندهاش طعم وداك الانتقال من انك دكتور في الجامعة المغربية لأنك مدرب في الجامعة الدولية كي كان هاد الإنتقال وانا انا بديت يعني بديت العمل ديالي في جامعة خارج المغرب سواء في المملكة العربية السعودية وفي الإمارات او في الولايات المتحدة الأمريكية اه ولا كنت في طور يعني ده,القال ي ال المربي مرب اللي ا خر المر ا ال,1.000,29.250000
3,كاين واحد التعقيد يراد به واحد المجموعة د الأشياء فمن تم يعني درت الدكتوراه في الفين بديت في الفين وتلطاش سنة الفين وتمنطاش ومن تم يعني كذلك يعني طورت القضية ديال الاستشارات المالية التدقيق الشرعي ديال الأيوفي هيئة المراجعة والمحاسبة المعاملات المالية الإسلامية,ل قيد اليا الر ر ال الا الس,1.000,26.600000
4,ديال ديال كل واحد يديها فراسو حسن ليه من المشاكل نديها فراسي ناس جاب نشوف لاخر نشوف راسي انا المصلحة ديالي لا ولا هاد كيقولك اودي المسؤولية ديالهم توقع على الدولة هوما اللي هادي وهادي لا انا كنقول الذي يحرك هاد الرقي الاجتماعي هو منظومة القيم التي نعيشها,ه رس المشاكل ا ر ا ظم القي,0.979,27.350000
5,غير معرفتش واش دابا نضحك ولا نكون معقول لي بغيت كنعرف يعني البرامج ديالك كيكون فيهم بزاف وانا معرفتش نكون المهم هو تعاودلي حكايتي عجبني العنوان شكرا كنهنئك واخا انا عرفت بان شهرزاد ديالي كانت كتحكي لشهر ايار لا بالعكس ماكاين تا مشكل توكلي على الله يالاه ها حنا غانمشيو,,1.000,28.150000
6,البناء اه بطبيعة الحال را الاعتدال مطلوب يعني ميمكنلناش نتصورو يعني اي بناء اولا توجه نحو التنمية بدون اعتدال لان لا الغلو ويضر والتراخي عاوتاني ايضا الغلو الدفاع مضر والتراخي ايضا مضر وبالتالي ان الاعتدال اساسي حنا المغرب منذ ثلاثة وتسعين والتوجه ديال المغرب في علاقة يعني القطع,بة الا اتدال ملب ربناء ا اعتدال ل اعتدال المرب من,0.979,27.650000
7,ديالكم حاولوا ما امكن تاخدوا مية فالمية ها حنا دابا وجدنا الأرض ديالنا اللي غادي ترتاح حتى الوقيتة ديال الربيع اللي خصنا نبداو نغرسوه الحاجة لي مهمة بزاف خصنا لي خصنا ناخدوهم هما الحمد لله دابا ولاو عندنا هاد فالمغرب للأسف مازال ما عندناش لي مية فالمية مغربيين ولكن اه نقدرو نلقاو لي,الي م اول,1.000,27.050000
8,كتخلص رسوم ديال التسجيل والتحفيض الى غير ذلك كتعلم ادارة الضرائب لأن كاين رسوم تسجيل كتأدى تاهو راه بمثابة اعلام واشعار الى غير ذلك ذلك اه اه كدلك خصنا نعلمو السلطة متلا ربما سكن ثانوي ولى سكن رئيسي وخا لأن الضريبة تختلي انا غادي نمشي نعلم فين غادي قلتي الجماعة نعم كاينة الجماعات وكاينة ادارة,,1.000,30.000000
9,مكاين تا واحد القيمة ديالو شحال غادي طيح تقمارت ولا ماشي تاقمرت معمرو يولي زيرو هدا النفط معمرو يولي زيرو ايجي شي واحد يقولك هادوك ديال ابريل الفين وعشرين كانوا هادوك هداك ماشي البرميل ديال النفط يعني اداة مقامرة او كيهيمنو عل الميدان المالي اليوم مع الأسف,دا مقامر ميدا الما ال ال,1.000,27.400000



── labeled_linagora_cs_arabize ──


,reference,prediction,sample_wer,duration_s
0,و الله لا وراس لا و الله مقتنع و الله اسمعني س ديجا,قمزلت مقتنع ليتي تحب تبطل جسبا يقعد معايا,1.000,7.776000
1,استفتاء,stifte,1.000,1.264000
2,يكري كرهبة,يكري كاربة,0.500,0.880000
3,عبد الله مشى سال طفلة,عبضنا شيسلطفلة,1.000,1.528000
4,هاذي غلطتي مش غلطتك,هذي غالتي مش غالت,0.750,2.216000
5,بودكاستينق يت ماكس ماي داي على خاطر فمة عباد تسمع فيك و يهمها لي تحكي فيه و كل و ذاي كير اباوت يو و يخمموا معناها,ا ميداي على خاطر فما بيت تسمع فيك ويهمها للي تحكي فيه وكل ي كار ا يو ويخمو معناها,0.692,9.560000
6,يطفيو الضو خلي قطر ولا الامارات تهز حتى كاس الحومة توا تشوف دنيا كيفاش تتقلب,يطفيو الضوء خلي قطر وا إلا الإمارات تهز حتى كاسالحومه توا تشوف الدنيا كيفاش تتقلب,0.467,6.424000
7,عبد الله سال لمرا في القيشي,عبدالله سال لمعا في اليشي,0.667,1.848000
8,كله كوم و ها قصر العداله الجديد الي تبنا كوم آخر الي هو خوك زكربرك ربي يفضله بنلنا ها القصر هذايا البلو,كل كوم ه قصر العدالة الجديدة اللي تبناء كوم آخر اللي هو خوك زوكر بارق ربي فضلو بنانها القسر هذيا البلو,0.682,8.712000
9,الي هو فيسبوك,اللي هو فيسبوك,0.333,1.056000



── labeled_linagora_raw ──


,reference,prediction,sample_wer,duration_s
0,و الله لا وراس لا و الله مقتنع و الله اسمعني c'إي déjà,قمزلت مقتنع ليتي تحب تبطل جسبا يقعد معايا,1.000,7.776000
1,استفتاء,stifte,1.000,1.264000
2,يكري كرهبة,يكري كاربة,0.500,0.880000
3,عبد الله مشى سأل طفلة,عبضنا شيسلطفلة,1.000,1.528000
4,هذي غلطتي موش غلطتك,هذي غالتي مش غالت,0.750,2.216000
5,podcasting it makes my day على خاطر فمة عباد تسمع فيك و يهمها لي تحكي فيه و كل و they care about you و يخمموا معناها,ا ميداي على خاطر فما بيت تسمع فيك ويهمها للي تحكي فيه وكل ي كار ا يو ويخمو معناها,0.731,9.560000
6,يطفيو الضو خلي قطر ولا الامارات تهز حتى كأس الحومة توا تشوف دنيا كيفاش تتقلب,يطفيو الضوء خلي قطر وا إلا الإمارات تهز حتى كاسالحومه توا تشوف الدنيا كيفاش تتقلب,0.467,6.424000
7,عبد الله سأل لمرا في القيشي,عبدالله سال لمعا في اليشي,0.833,1.848000
8,كله كوم و ها قصر العداله الجديد إلي تبنا كوم آخر إلي هو خوك زكربرك ربي يفضله بنلنا ها القصر هذايا البلو,كل كوم ه قصر العدالة الجديدة اللي تبناء كوم آخر اللي هو خوك زوكر بارق ربي فضلو بنانها القسر هذيا البلو,0.682,8.712000
9,إلي هو فيسبوك,اللي هو فيسبوك,0.333,1.056000


## 7 · Worst predictions

In [14]:
display_worst(all_result_dfs, TOP_N_WORST)


── labeled_algerian — top 15 worst predictions ──


,reference,prediction,sample_wer,duration_s
83,يعطيك الصحة مرة أخرى وثاني للناس هنا معظم متابعينا تقدر تقول عندهم خلفية تقنية لكن كاين ما شاء الله الناس في المجالات متخصصة وحداخرى نحاولو علابيها رانا نحاولو النوعو ثاني المحتوى في هذا الموسم ولي يهتم ثاني بلاك في هذا الموضوع ولكن من ناحية التقنية كاين الحلقة سجلناها مع الأخ,علي الاقل اعطيهم امتيازات اعطيهم حوايج اللي يقدرو يجوك علي بهان المدن داخله باش نقدروا نديروا قطب مليح قال فجنوبب يعودوا يجو يقراوا عادنا تعالجلفه ورقله وماشانونزيد لك كارثه وحد اخري هيتسرا سيكم بعد هذي فا يسما في التدرج يسما باشتلحق ماستر بعد المشكل كلقالي روحوا الدكتورا سما الاشكال ا نهاره القناس راحو الدكتورا كيتدير نتا جامعاه بارتوي تمها شول الاشكال يدخلو فيها المحسوبيه والجهويه,1.280,28.432750
378,إيه لأنو حاجة مليحة أنو الناس ستفادت من الغاز حاجة مليحة أكيد أنو ناس يلحقلها الغاز للدار آه حاجة أأ وإنما الإشكال أنو إلا نتا المداخيل تاعك مربوطة بالغاز مبعد غدوة تولي تستهلك قاع ال المنتج تاعك هذا تاع الغاز ماركش بلد بترولي راك في وجه أزمة كبيرة وبالتالي حنا,كبرت في الجزائر وكانو خرين دحد عملالجزائر وخرى تعاونمع الحكومة الجزائرية ف ستينات هذا الشي اللي عاوني وعاونونيحتى تعلمت ومشيت عملت اختصاص في جامعة بيت لليقدة الدوائية لان كاعنديهذكالمور العملية كنت نعمل كمختصفاليقة الدوائية لكنكان خاصني تعمق اكثر ودراساتعملت ما قادة دوائية في جامعة بار وبديت نعمل من بعد مشيتمن مستشفى الباريز للمستشفى,1.060,46.914750
2,تكون التجربة ما عندهاش طعم وداك الانتقال من انك دكتور في الجامعة المغربية لأنك مدرب في الجامعة الدولية كي كان هاد الإنتقال وانا انا بديت يعني بديت العمل ديالي في جامعة خارج المغرب سواء في المملكة العربية السعودية وفي الإمارات او في الولايات المتحدة الأمريكية اه ولا كنت في طور يعني ده,القال ي ال المربي مرب اللي ا خر المر ا ال,1.000,29.250000
3,كاين واحد التعقيد يراد به واحد المجموعة د الأشياء فمن تم يعني درت الدكتوراه في الفين بديت في الفين وتلطاش سنة الفين وتمنطاش ومن تم يعني كذلك يعني طورت القضية ديال الاستشارات المالية التدقيق الشرعي ديال الأيوفي هيئة المراجعة والمحاسبة المعاملات المالية الإسلامية,ل قيد اليا الر ر ال الا الس,1.000,26.600000
5,غير معرفتش واش دابا نضحك ولا نكون معقول لي بغيت كنعرف يعني البرامج ديالك كيكون فيهم بزاف وانا معرفتش نكون المهم هو تعاودلي حكايتي عجبني العنوان شكرا كنهنئك واخا انا عرفت بان شهرزاد ديالي كانت كتحكي لشهر ايار لا بالعكس ماكاين تا مشكل توكلي على الله يالاه ها حنا غانمشيو,,1.000,28.150000
7,ديالكم حاولوا ما امكن تاخدوا مية فالمية ها حنا دابا وجدنا الأرض ديالنا اللي غادي ترتاح حتى الوقيتة ديال الربيع اللي خصنا نبداو نغرسوه الحاجة لي مهمة بزاف خصنا لي خصنا ناخدوهم هما الحمد لله دابا ولاو عندنا هاد فالمغرب للأسف مازال ما عندناش لي مية فالمية مغربيين ولكن اه نقدرو نلقاو لي,الي م اول,1.000,27.050000
8,كتخلص رسوم ديال التسجيل والتحفيض الى غير ذلك كتعلم ادارة الضرائب لأن كاين رسوم تسجيل كتأدى تاهو راه بمثابة اعلام واشعار الى غير ذلك ذلك اه اه كدلك خصنا نعلمو السلطة متلا ربما سكن ثانوي ولى سكن رئيسي وخا لأن الضريبة تختلي انا غادي نمشي نعلم فين غادي قلتي الجماعة نعم كاينة الجماعات وكاينة ادارة,,1.000,30.000000
9,مكاين تا واحد القيمة ديالو شحال غادي طيح تقمارت ولا ماشي تاقمرت معمرو يولي زيرو هدا النفط معمرو يولي زيرو ايجي شي واحد يقولك هادوك ديال ابريل الفين وعشرين كانوا هادوك هداك ماشي البرميل ديال النفط يعني اداة مقامرة او كيهيمنو عل الميدان المالي اليوم مع الأسف,دا مقامر ميدا الما ال ال,1.000,27.400000
10,ولا غير با يشري رقيعة فيها غير الما يما نعرفو بلا ما يكون شي ايلا كان راه غي بالما غادي يتحيد ولا ناخدو القهوة نخويوها عليه غي غادي تمسح ديك الساعة مغادي يبقى فيه حتى شي حاجة دابا ماشي غير هدا من غير لي كيكونو لا حتى هما فيهم دابا رجع فكلشي حيت كنلقاو كيجيو مخيطين,,1.000,28.050000
11,اهاه بالعكس بالعكس نسخة ثقيل وفبزاف د الأحيان الشخصية القوية كتخلع كتخلع الرجال كتخلع وحتى هداكشي دائما كاين اهم داكشي لي كتعكسيه الواحد لي كيكون باغي يكون هو مقادرش فمكيبغيكش حيت نتي كتمتلي واحد الحاجة اللي هو باغيها اللي ما كرهش يكون هو فبلاصتو كيكون هو عندو ديك الشخصية ويقدر هو ياخد ديك القرار,ا اتلع ق ال اقرار,1.000,26.400000



── labeled_linagora_cs_arabize — top 15 worst predictions ──


,reference,prediction,sample_wer,duration_s
3040,نرمالمن سنسي يقراو امم,قرأالله تسلفي وعلى على م الحضاير متعرضين للمدرسة ني نعطيك معها في بريا في اسرين للان مسكر بالحجرة,4.500,10.672000
1454,الترموماتر,et er mometr,3.000,1.464000
3109,البوفي,eل l fait,3.000,0.696000
4413,مرحبا,بل ش مسوس,3.000,1.680000
1995,عملوش بخاطر,ملوش ي ب تيبي خاطر,2.500,2.104000
428,بالسلامة,بس شامة,2.000,0.712000
598,الفوتنغ,ال footing,2.000,1.040000
635,مشحاح,مش حاح,2.000,0.920000
644,العرس,إلى عنس,2.000,0.616000
1118,اترفار,a traver,2.000,0.752000



── labeled_linagora_raw — top 15 worst predictions ──


,reference,prediction,sample_wer,duration_s
3040,نرمالمن سنسي يقراو إمم,قرأالله تسلفي وعلى على م الحضاير متعرضين للمدرسة ني نعطيك معها في بريا في اسرين للان مسكر بالحجرة,4.500,10.672000
1454,الترموماتر,et er mometr,3.000,1.464000
3109,البوفي,eل l fait,3.000,0.696000
4413,مرحبا,بل ش مسوس,3.000,1.680000
1995,عملوش بخاطر,ملوش ي ب تيبي خاطر,2.500,2.104000
428,بالسلامة,بس شامة,2.000,0.712000
598,الفوتنغ,ال footing,2.000,1.040000
635,مشحاح,مش حاح,2.000,0.920000
644,العرس,إلى عنس,2.000,0.616000
1118,أترفار,a traver,2.000,0.752000


## 8 · Unlabelled inspection

In [16]:
unlabelled_result_dfs = run_unlabelled_splits(
    benchmark, UNLABELLED_SPLITS, infer_fn, OUTPUT_DIR, 'omniasr_ctc_1b_unlabelled_results'
)


  Running (unlabelled): unlabeled_youtube  (398 samples)


Inferring:   0%|          | 0/50 [00:00<?, ?batch/s]

  ✓ CSV saved → /home/ala/TunisianDialogSystem/outputs/asr_benchmark_results/omniasr_ctc_1b_results/unlabeled_youtube_omniasr_ctc_1b_unlabelled_results.csv

✅ All unlabelled splits done.


In [17]:
unlabelled_result_dfs = run_unlabelled_splits(
    benchmark,
    UNLABELLED_SPLITS,
    infer_fn,
    OUTPUT_DIR,
    FILE_SUFFIX,
    resume=RESUME,
)


  Running (unlabelled): unlabeled_youtube  (398 samples)


Inferring:   0%|          | 0/50 [00:00<?, ?batch/s]

  ✓ CSV saved → /home/ala/TunisianDialogSystem/outputs/asr_benchmark_results/omniasr_ctc_1b_results/unlabeled_youtube_omniasr_ctc_1b_results.csv

✅ All unlabelled splits done.


In [ ]:
display_bulk_predictions(unlabelled_result_dfs)

## 9 · Summary

In [ ]:
summary_df = display_summary(
    summary_rows, OUTPUT_DIR, 'omniasr_ctc_1b', 'omniasr_ctc_1b_results'
)

In [ ]:
if summary_df is not None:
    plot_wer_cer(summary_df, OUTPUT_DIR, 'omniasr_ctc_1b', 'omniasr_ctc_1b_results')